In [2]:
from pathlib import Path
from paths.pathsval import (project_root, runends_directory)
from download_data.get_downloads_info import (run_end, save_run_ends)
import pandas as pd


In [5]:
base_project_path = project_root()
bioproject = "168994"  # Example bioproject ID
project = "GSM"  # Options: "GSE" or "SRA"
gse_test = "GSM461176,GSM461177,GSM461178,GSM461179,GSM461180,GSM4681,GSM461182"  # Comma-separated GSE or GSM list for testing
bpdir = base_project_path / "data"  / bioproject

print(f"Bioproject directory: {bpdir}")

sra_files_dir = bpdir / "sra_files"
gds_files_dir = bpdir / "gds_files"

sra_info_path = sra_files_dir / "sra_runinfo.tsv"
gds_info_path = gds_files_dir / "gse_gsm.tsv"

if not sra_info_path.is_file():
    raise FileNotFoundError(f"SRA runinfo file not found at {sra_info_path}")

print (f"Reading SRA runinfo from {sra_info_path}")
    

if project == "GSE":
    try:
        sra_df = pd.read_csv(sra_info_path, sep="\t")
        if sra_df.empty:
            raise ValueError("SRA runinfo file is empty.")
        if 'Run' not in sra_df.columns or 'LibraryLayout' not in sra_df.columns:
            raise ValueError("SRA runinfo file does not contain 'Run' or 'LibraryLayout' column.")
    except Exception as e:
        raise ValueError(f"Error reading SRA runinfo file: {e}")

    try:
        df_gse_gsm = pd.read_csv(gds_info_path, sep=",")
        if df_gse_gsm.empty:
            raise ValueError("GDS info file is empty.")
        if 'GSE' not in df_gse_gsm.columns or 'GSM' not in df_gse_gsm.columns:
            raise ValueError("GDS info file does not contain 'GSE' or 'GSM' column.")
    except Exception as e:
        raise ValueError(f"Error reading GDS info file: {e}")

   
    for gse in gse_test.split(","):
       gse = gse.strip()
       gsm_list = df_gse_gsm[df_gse_gsm['GSE'] == gse]['GSM'].tolist()
       sra_subset = sra_df[sra_df['SampleName'].isin(gsm_list)]
       if sra_subset.empty:
           print(f"No matching GSM entries found in SRA runinfo for {gse}. Skipping.")
           continue

       single, paired = run_end(sra_subset)
       output_path = runends_directory(base_project_path, bioproject)
       paired_path, single_path = save_run_ends(paired, single, output_path, gse)

       print(f"Processed {gse}:")
       print(f"  Paired-end runs saved to: {paired_path}")
       print(f"  Single-end runs saved to: {single_path}")


if project == "GSM":
    try:
        sra_df = pd.read_csv(sra_info_path, sep="\t")
        if sra_df.empty:
            raise ValueError("SRA runinfo file is empty.")
        if 'Run' not in sra_df.columns or 'LibraryLayout' not in sra_df.columns:
            raise ValueError("SRA runinfo file does not contain 'Run' or 'LibraryLayout' column.")
    except Exception as e:
        raise ValueError(f"Error reading SRA runinfo file: {e}")

    gsm_list = [gsm.strip() for gsm in gse_test.split(",")]
    sra_subset = sra_df[sra_df['SampleName'].isin(gsm_list)]
    if sra_subset.empty:
        raise ValueError("No matching GSM entries found in SRA runinfo.")

    paired, single = run_end(sra_subset)
    output_path = runends_directory(base_project_path, bioproject)
    paired_path, single_path = save_run_ends(paired, single, output_path, gsm_list[0] + "-" + gsm_list[-1])
        
    print(f"Processed selected GSMs:")
    print(f"  Paired-end runs saved to: {paired_path}")
    print(f"  Single-end runs saved to: {single_path}")

elif project == "SRA":
    try:
        sra_df = pd.read_csv(sra_info_path, sep="\t")
        if sra_df.empty:
            raise ValueError("SRA runinfo file is empty.")
        if 'Run' not in sra_df.columns or 'LibraryLayout' not in sra_df.columns:
            raise ValueError("SRA runinfo file does not contain 'Run' or 'LibraryLayout' column.")
    except Exception as e:
        raise ValueError(f"Error reading SRA runinfo file: {e}")
         
    single, paired = run_end(sra_df)
    output_path = runends_directory(base_project_path, bioproject)
    paired_path, single_path = save_run_ends(paired, single, output_path, bioproject)
    
    print(f"Processed SRA project {bioproject}:")
    print(f"  Paired-end runs saved to: {paired_path}")
    print(f"  Single-end runs saved to: {single_path}")


Bioproject directory: C:\Users\pdmpe\OneDrive\Documentos\GitHub\RbohB-DE\data\168994
Reading SRA runinfo from C:\Users\pdmpe\OneDrive\Documentos\GitHub\RbohB-DE\data\168994\sra_files\sra_runinfo.tsv
Single-end reads:            SRR
379  SRR031708
380  SRR031709
381  SRR031710
382  SRR031711
383  SRR031712
384  SRR031713
389  SRR031718
390  SRR031719
391  SRR031720
392  SRR031721
393  SRR031722
394  SRR031723
399  SRR031728
400  SRR031729
Paired-end reads:            SRR
385  SRR031714
386  SRR031715
387  SRR031716
388  SRR031717
395  SRR031724
396  SRR031725
Total Single-end: 14
Total Paired-end: 6
Processed selected GSMs:
  Paired-end runs saved to: C:\Users\pdmpe\OneDrive\Documentos\GitHub\RbohB-DE\results\168994\runends\GSM461176-GSM461182_paired_end_runs.tsv
  Single-end runs saved to: C:\Users\pdmpe\OneDrive\Documentos\GitHub\RbohB-DE\results\168994\runends\GSM461176-GSM461182_single_end_runs.tsv
